## Computational Fluid Dynamics (CFD)

### Lid-driven cavity — vectorized solver

This notebook is the **vectorized** counterpart of `navier-stokes.ipynb`.
It solves the exact same Navier-Stokes lid-driven-cavity problem (pressure and
streamfunction via SOR, momentum via explicit Euler/RK1), but every per-cell
Python loop has been replaced by NumPy array operations:

* momentum, velocity correction, BCs and contour fields are computed by array
  slicing (arithmetically identical to the scalar version);
* the pressure / streamfunction SOR sweeps use **red-black ordering**, the
  vectorizable equivalent of the original sequential SOR — it converges to the
  same flow field.

At 128×128 / 100 steps this runs in a few seconds instead of several minutes
(~80× faster). The cells below are identical to the original notebook except for
the import.

### System setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys

In [2]:
sys.path.append('../src/')
from flow_simulation_vectorized import simulate_flow

### Parameters

#### Grid parameters

In [11]:
Nxc = 32
Nyc = 32

#### Simulation parameters

In [12]:
Re = 100   # Reynolds Number
tf = 10.0   # Final Simulation Time
SOR = 1.5   # SOR Factor
n_iter = 10 # Max. # of iterations

# Time step. The explicit scheme is only stable if dt respects the CFL limit
# dt < dx / U_lid = (1/Nxc) / 1. At Nxc=128 that means dt < 0.0078, so dt=0.01
# diverges on long runs (it blows up around t~36). dt=0.004 (CFL~0.5) is stable
# for the full tf=50.
#   NOTE: 50/0.004 = 12,500 steps (~8-9 min). Watch the convergence plot below:
#   the flow reaches steady state well before t=50, so you can lower tf to stop
#   earlier. The solver also warns and stops safely if dt is left too large.
dt = 0.001  # Time Step


#### Boundary conditions

In [13]:
Un = 1.0 # upper boundary
Us = 0.0 # bottom plate
Vw = 0.0 # left plate
Ve = 0.0 # right plate

#### Initial Conditions

In [14]:
u = np.zeros((Nxc+1, Nyc+2))
v = np.zeros((Nxc+2, Nyc+1))

### Run the simulation

In [ ]:
%%time

# return_diagnostics=True also returns a dict with the convergence history
# (`error`, `umax`), the CFL number, and divergence info if it blew up.
X, Y, STREAMLINE, u_contour, v_contour, p_contour, vorticity, diagnostics = (
    simulate_flow(u, v, Nxc, Nyc, dt, tf, n_iter, SOR, Re, Us, Un, Vw, Ve,
                  return_diagnostics=True)
)

Running iteration number 1248 of 10000

In [ ]:
# If the plots below don't work, try installing latex as shown below...
# !pip install latex

### Visualize

In [ ]:
fig, ax = plt.subplots(3, 2, figsize=(16, 20), dpi=100)

ax = ax.ravel()

ax[0].quiver(
    np.flipud(np.rot90(u_contour)),
    np.flipud(np.rot90(v_contour))
)

ax[1].contour(X, Y, STREAMLINE, 100)
ax[2].contour(np.rot90(np.fliplr(u_contour)), 100)
ax[3].contour(np.rot90(np.fliplr(v_contour)), 100)
ax[4].contour(np.rot90(np.fliplr(p_contour)), 100)
ax[5].contour(np.rot90(np.fliplr(vorticity)), 100)

fig.tight_layout()

### Convergence & stability

`diagnostics` carries the per-step history. `error` is the pressure-iteration
RMSE (how much the pressure field still changes each step) and `umax` is the
maximum velocity magnitude — both should flatten out once the flow reaches
steady state. If the run had diverged, `diagnostics['diverged_step']` would be
set and the solver would have stopped and warned instead of producing blank
charts.

In [ ]:
print(f"convective CFL      : {diagnostics['cfl']:.2f}")
print(f"diffusive dt limit  : {diagnostics['dt_diffusive_limit']:.4g}")
print(f"steps completed     : {diagnostics['steps_completed']}")
print(f"diverged at step    : {diagnostics['diverged_step']}")

err = diagnostics['error']
umax = diagnostics['umax']
steps = np.arange(1, len(err) + 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].semilogy(steps, err)
ax[0].set_xlabel('time step'); ax[0].set_ylabel('pressure RMSE')
ax[0].set_title('Convergence (lower = steadier)')
ax[1].plot(steps, umax)
ax[1].set_xlabel('time step'); ax[1].set_ylabel('max |u|')
ax[1].set_title('Max velocity (flat = steady state)')
fig.tight_layout()